# 04 — Migrationseffekter: JSFS, $F_{ST}$ og asymmetrisk migration

I denne notebook vil jeg se nærmere ind i to-ø modellens egenskaber: jeg beregner joint site frequency spectrum (JSFS), genetisk differentiation ($F_{ST}$) og undersøger hvad der sker, når migration er asymmetrisk eller når jeg tilføjer en tredje 'ghost' population.

**Spørgsmål jeg gerne vil undersøger:**

- Hvad siger JSFS om private vs. delte varianter under varierende $M$?
- Hvordan ændrer $F_{ST}$ sig som funktion af $M$?
- Hvad sker der med TMRCA og JSFS for asymmetrisk migration $M_{12} \neq M_{21}$?
- Hvad sker der, hvis jeg tilføjer en ghost population?

In [ ]:
from phasic import Graph, with_ipv, StateIndexer, Property
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%config InlineBackend.figure_format = 'svg'
np.random.seed(42)
sns.set_palette('tab10')
plt.rcParams['figure.dpi'] = 120


In [ ]:
# Genimporter build_two_island fra forrige notebook
def build_two_island(n_pop1, n_pop2, M, N=1.0):
    @with_ipv([n_pop1, n_pop2])
    def two_island_cb(state):
        transitions = []
        i, j = state[0], state[1]
        if i >= 2:
            new = state.copy(); new[0] -= 1
            transitions.append((new, i*(i-1)/2/N))
        if j >= 2:
            new = state.copy(); new[1] -= 1
            transitions.append((new, j*(j-1)/2/N))
        if i >= 1:
            new = state.copy(); new[0] -= 1; new[1] += 1
            transitions.append((new, i*M/2))
        if j >= 1:
            new = state.copy(); new[1] -= 1; new[0] += 1
            transitions.append((new, j*M/2))
        return transitions
    return Graph(two_island_cb)

## 1. Joint site frequency spectrum (JSFS)

JSFS-matricen $F_{ij}$ angiver den forventede grenlængde for genealogiske linjer med $i$ efterkommere i pop1 og $j$ efterkommere i pop2.

I to-ø modellen med StateIndexer bruger jeg phasic's reward-system: reward-vektoren for celle $(i,j)$ tæller antallet af linjer i tilstande der bidrager til netop den celle.

**Fortolkning af JSFS:**

- **Kanten** $F_{i,0}$ og $F_{0,j}$: private varianter (kun i en pop)
- **Indre celler** $F_{i,j}, i>0, j>0$: delte varianter (begge pops)
- Lav $M$ → dominans af private varianter
- Høj $M$ → øget andel delte varianter

In [ ]:
# JSFS via StateIndexer og reward-transformation
def compute_jsfs(n1, n2, M, N=1.0):
    """
    Beregn expected JSFS for two-island model.
    Returnerer matrix F[i,j] = E[grenlængde med i efterkommere i pop1, j i pop2].
    """
    g = build_two_island(n1, n2, M=M, N=N)
    states = g.states()  # shape: (n_vertices, 2) — [pop1, pop2]
    
    n_total = n1 + n2
    jsfs = np.zeros((n1 + 1, n2 + 1))
    
    # Beregn reward for hvert (i,j)-par
    for i in range(n1 + 1):
        for j in range(n2 + 1):
            if i == 0 and j == 0:
                continue
            # Reward: 1 for tilstande (i, j), 0 ellers
            # En tilstand i grafen er [pop1_count, pop2_count]
            # Vi søger tilstande med præcis pop1=i og pop2=j
            reward = np.array([
                1.0 if (s[0] == i and s[1] == j) else 0.0
                for s in states
            ])
            # E[grenlængde] = E[tid brugt i tilstande med (i,j)]
            jsfs[i, j] = g.expectation(reward)
    
    return jsfs, g

# Test for n1=3, n2=1, M=2
jsfs_test, g_test = compute_jsfs(3, 1, M=2.0)
print("JSFS for n1=3, n2=1, M=2:")
df_jsfs = pd.DataFrame(jsfs_test,
                        index=[f'pop1={i}' for i in range(4)],
                        columns=[f'pop2={j}' for j in range(2)])
print(df_jsfs.round(4))
print(f"\nSum (= E[T_total]): {jsfs_test.sum():.4f}")

In [ ]:
# JSFS for varierende M — heatmaps
M_jsfs = [0.2, 1.0, 5.0, 20.0]
n1, n2 = 4, 4

fig, axes = plt.subplots(1, len(M_jsfs), figsize=(15, 4))

for ax, M in zip(axes, M_jsfs):
    jsfs, _ = compute_jsfs(n1, n2, M=M)
    # Ekskluder (0,0) og normaliser
    jsfs_plot = jsfs.copy()
    jsfs_plot[0, 0] = np.nan
    
    im = ax.imshow(jsfs_plot, cmap='YlOrRd', aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_xlabel('Pop2 efterkommere')
    ax.set_ylabel('Pop1 efterkommere')
    ax.set_title(f'JSFS (M={M})')
    ax.set_xticks(range(n2 + 1))
    ax.set_yticks(range(n1 + 1))
    
    # Annotér celler
    for i in range(n1 + 1):
        for j in range(n2 + 1):
            if not np.isnan(jsfs_plot[i, j]):
                ax.text(j, i, f'{jsfs_plot[i,j]:.2f}',
                        ha='center', va='center', fontsize=7,
                        color='white' if jsfs_plot[i,j] > jsfs_plot.max()*0.6 else 'black')

plt.suptitle('Joint Site Frequency Spectrum for varierende M (n=4+4)', y=1.02)
plt.tight_layout()
plt.show()

print("Observation:")
print("- Lav M: JSFS domineres af kanterne (private varianter F[i,0] og F[0,j]).")
print("- Høj M: Intern JSFS-masse øges (delte varianter F[i,j] med i,j>0).")
print("- Diagonalen (i=j) stiger med M — varianter opstår symmetrisk i begge pops.")

In [ ]:
# Kvantificer private vs. delte varianter som funktion af M
M_range = np.logspace(-1.5, 1.5, 20)
n1, n2 = 3, 3
private_frac = []
shared_frac = []

for M in M_range:
    jsfs, _ = compute_jsfs(n1, n2, M=M)
    total = jsfs.sum()
    # Private: kun kanten
    private = jsfs[1:, 0].sum() + jsfs[0, 1:].sum()
    shared = total - private
    private_frac.append(private / total * 100)
    shared_frac.append(shared / total * 100)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(M_range, private_frac, 'o-', label='Private varianter (%)', lw=2)
ax.semilogx(M_range, shared_frac, 's-', label='Delte varianter (%)', lw=2)
ax.axhline(50, color='gray', linestyle='--', lw=1, alpha=0.5)
ax.set_xlabel('Migrationsrate M')
ax.set_ylabel('Andel af total grenlængde (%)')
ax.set_title('Private vs. delte varianter som funktion af M')
ax.legend()
plt.tight_layout()
plt.show()
print(f"M ved 50/50 split: approx M ≈ {M_range[np.argmin(np.abs(np.array(private_frac)-50))]:.2f}")

## Genetisk differentiation: $F_{ST}$ via phase-type

$F_{ST}$ måler genetisk differentiation mellem to populationer:

$$F_{ST} = 1 - \frac{\mathbb{E}[T_{\text{within}}]}{\mathbb{E}[T_{\text{between}}]},$$

hvor $T_{\text{within}}$ er TMRCA for to linjer samplet fra *samme* population
og $T_{\text{between}}$ er TMRCA for to linjer fra *forskellige* populationer.

Relationerne er:
- $\mathbb{E}[T_{\text{within}}] = 1 + \frac{1}{M}$ (begge fra pop1 → pop1)
- $\mathbb{E}[T_{\text{between}}] = 1 + \frac{2}{M}$ (én fra hver)

In [ ]:
M_range = np.logspace(-2, 2, 80)
fst_values = []
E_within_vals = []
E_between_vals = []

for M in M_range:
    g_within = build_two_island(2, 0, M=M)   # begge i pop1
    g_between = build_two_island(1, 1, M=M)  # én i hver
    
    E_w = g_within.expectation()
    E_b = g_between.expectation()
    
    fst = 1 - E_w / E_b
    fst_values.append(fst)
    E_within_vals.append(E_w)
    E_between_vals.append(E_b)

# Analytisk: FST = (1/(1+M)) / (1 + 1/(1+M)) = 1/(1+2M)
# (fra standard FST-formel for stepping-stone modellen)
fst_ana = 1 / (1 + 2 * M_range)  # klassisk approksimation

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].loglog(M_range, fst_values, '-', label='Phasic FST', lw=2)
axes[0].loglog(M_range, fst_ana, '--', label='1/(1+2M)', lw=1.5, alpha=0.8)
axes[0].set_xlabel('Migrationsrate M')
axes[0].set_ylabel('FST')
axes[0].set_title('FST som funktion af M')
axes[0].legend()

axes[1].semilogx(M_range, E_within_vals, label='E[T_within]', lw=2)
axes[1].semilogx(M_range, E_between_vals, label='E[T_between]', lw=2)
axes[1].set_xlabel('Migrationsrate M')
axes[1].set_ylabel('E[TMRCA]')
axes[1].set_title('TMRCA within vs. between pops')
axes[1].legend()

plt.tight_layout()
plt.show()
print("Observation:")
print("- FST ≈ 1 for meget lav M (næsten fuld differentiation).")
print("- FST → 0 for høj M (panmiksi, ingen differentiation).")
print("- Phasic-estimat og klassisk formel 1/(1+2M) er næsten identiske.")

## Asymmetrisk migration: $M_{12} \neq M_{21}$

I mange biologiske scenarier er migration asymmetrisk — fx er en øpopulation primært modtager fra fastlandet. Vi udvider modellen med to uafhængige migrationsrater $M_{12}$ og $M_{21}$.

**Eksperiment:** Hvad sker der med TMRCA og FST, hvis $M_{12} \neq M_{21}$?

In [ ]:
# Asymmetrisk two-island model
def build_asymmetric_two_island(n_pop1, n_pop2, M12, M21, N=1.0):
    """
    Two-island med asymmetrisk migration.
    M12: migrationsrate pop1 → pop2
    M21: migrationsrate pop2 → pop1
    """
    @with_ipv([n_pop1, n_pop2])
    def asym_cb(state):
        transitions = []
        i, j = state[0], state[1]
        
        if i >= 2:
            new = state.copy(); new[0] -= 1
            transitions.append((new, i*(i-1)/2/N))
        if j >= 2:
            new = state.copy(); new[1] -= 1
            transitions.append((new, j*(j-1)/2/N))
        if i >= 1:  # Migration pop1 → pop2
            new = state.copy(); new[0] -= 1; new[1] += 1
            transitions.append((new, i * M12 / 2))
        if j >= 1:  # Migration pop2 → pop1
            new = state.copy(); new[1] -= 1; new[0] += 1
            transitions.append((new, j * M21 / 2))
        
        return transitions
    
    return Graph(asym_cb)

# Test: total migration M_total = M12 + M21 = 2, men asymmetrisk
M_total = 2.0
asym_scenarios = [
    (2.0, 2.0, 'Symmetrisk (M12=M21=2)'),
    (3.0, 1.0, 'Asymmetrisk (M12=3, M21=1)'),
    (1.0, 3.0, 'Asymmetrisk (M12=1, M21=3)'),
    (3.8, 0.2, 'Stærkt asymmetrisk (M12=3.8, M21=0.2)'),
    (0.2, 3.8, 'Stærkt asymmetrisk (M12=0.2, M21=3.8)'),
]

# TMRCA for to linjer: begge i pop1
t_plot = np.linspace(0, 15, 400)
fig, ax = plt.subplots(figsize=(9, 5))

for M12, M21, label in asym_scenarios:
    g = build_asymmetric_two_island(2, 0, M12=M12, M21=M21)
    ax.plot(t_plot, g.pdf(t_plot), label=f'{label}\nE={g.expectation():.2f}', lw=1.8)

ax.set_xlabel('TMRCA')
ax.set_ylabel('Tæthed')
ax.set_title('PDF for TMRCA med asymmetrisk migration (begge linjer i pop1)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print("Observation:")
print("- Fordelingen er IKKE symmetrisk i M12 og M21.")
print("- Høj M21 (pop2→pop1) = hurtig tilbagemigration → ligner isolation mere.")
print("- Høj M12 (pop1→pop2) = hurtig udvandring → linjer i pop1 forenes hurtigere.")

In [ ]:
# Systematisk: varier M12 og M21 uafhængigt — 2D heatmap af E[TMRCA]
M12_vals = [0.2, 0.5, 1, 2, 5]
M21_vals = [0.2, 0.5, 1, 2, 5]

E_grid = np.zeros((len(M12_vals), len(M21_vals)))
Var_grid = np.zeros_like(E_grid)

for i, M12 in enumerate(M12_vals):
    for j, M21 in enumerate(M21_vals):
        g = build_asymmetric_two_island(2, 0, M12=M12, M21=M21)
        E_grid[i, j] = g.expectation()
        Var_grid[i, j] = g.variance()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, data, title in zip(axes, [E_grid, Var_grid], ['E[TMRCA]', 'Var[TMRCA]']):
    im = ax.imshow(data, cmap='YlOrRd', aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(M21_vals)))
    ax.set_yticks(range(len(M12_vals)))
    ax.set_xticklabels(M21_vals)
    ax.set_yticklabels(M12_vals)
    ax.set_xlabel('M21 (pop2 → pop1)')
    ax.set_ylabel('M12 (pop1 → pop2)')
    ax.set_title(title)
    for i in range(len(M12_vals)):
        for j in range(len(M21_vals)):
            ax.text(j, i, f'{data[i,j]:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if data[i,j] > data.max()*0.7 else 'black')

plt.suptitle('E[TMRCA] og Var[TMRCA] for asymmetrisk migration (n=2, begge i pop1)')
plt.tight_layout()
plt.show()
print("Observation:")
print("- E[TMRCA] er lavest i øverste højre hjørne (høj M12 OG M21).")
print("- M21 (returmigration) reducerer TMRCA mere end M12 for pop1-startede linjer.")
print("- Variansen er størst langs den venstre og nedre kant (lav migration).")

## 4. Ghost population: en ikke-samplet tredje deme

En ghost population er en population der påvirker genealogierne men aldrig er samplet. Jeg tilføjer en tredje population $G$ med migration til/fra pop1 og pop2. Linjer kan "parkere" i ghost-populationen, hvilket forlænger TMRCA for de sampledede populationer.

**Tilstand:** $(i, j, k)$ = antal linjer i pop1, pop2, ghost.
Jeg starter med alle linjer i pop1 (ingen i pop2 eller ghost).

In [ ]:
def build_three_pop(n1, n2, M12, M21, Mg1, Mg2, N=1.0):
    """
    Tre-populations model med ghost population G.
    
    Migration:
        M12, M21: between pop1 and pop2
        Mg1: ghost → pop1 (immigration ind i pop1)
        Mg2: ghost → pop2 (immigration ind i pop2)
        For simplifikation: emigration fra pop1/pop2 til ghost er symmetrisk.
    
    Tilstand: [pop1, pop2, ghost] = antal linjer i hver population.
    """
    @with_ipv([n1, n2, 0])  # start: alle i pop1, ingen i pop2 eller ghost
    def three_pop_cb(state):
        transitions = []
        i, j, k = state[0], state[1], state[2]  # pop1, pop2, ghost
        
        # Koalescens i pop1
        if i >= 2:
            new = state.copy(); new[0] -= 1
            transitions.append((new, i*(i-1)/2/N))
        # Koalescens i pop2
        if j >= 2:
            new = state.copy(); new[1] -= 1
            transitions.append((new, j*(j-1)/2/N))
        # Koalescens i ghost
        if k >= 2:
            new = state.copy(); new[2] -= 1
            transitions.append((new, k*(k-1)/2/N))
        
        # Migration pop1 ↔ pop2
        if i >= 1:
            new = state.copy(); new[0] -= 1; new[1] += 1
            transitions.append((new, i*M12/2))
        if j >= 1:
            new = state.copy(); new[1] -= 1; new[0] += 1
            transitions.append((new, j*M21/2))
        
        # Migration pop1 ↔ ghost
        if i >= 1:
            new = state.copy(); new[0] -= 1; new[2] += 1
            transitions.append((new, i*Mg1/2))
        if k >= 1:
            new = state.copy(); new[2] -= 1; new[0] += 1
            transitions.append((new, k*Mg1/2))
        
        # Migration pop2 ↔ ghost
        if j >= 1:
            new = state.copy(); new[1] -= 1; new[2] += 1
            transitions.append((new, j*Mg2/2))
        if k >= 1:
            new = state.copy(); new[2] -= 1; new[1] += 1
            transitions.append((new, k*Mg2/2))
        
        return transitions
    
    return Graph(three_pop_cb)

# Sammenlign: med og uden ghost
M_sym = 1.0
Mg_values = [0, 0.5, 2.0, 5.0]  # ghost-migrationsrate

t_plot = np.linspace(0, 20, 400)
fig, ax = plt.subplots(figsize=(9, 5))

g_no_ghost = build_two_island(2, 0, M=M_sym)
ax.plot(t_plot, g_no_ghost.pdf(t_plot), 'k--', lw=2, label=f'Ingen ghost (E={g_no_ghost.expectation():.2f})')

for Mg in [mg for mg in Mg_values if mg > 0]:
    g = build_three_pop(2, 0, M12=M_sym, M21=M_sym, Mg1=Mg, Mg2=Mg)
    ax.plot(t_plot, g.pdf(t_plot), label=f'Ghost M_g={Mg} (E={g.expectation():.2f})', lw=1.8)

ax.set_xlabel('TMRCA')
ax.set_ylabel('Tæthed')
ax.set_title(f'Effekt af ghost population på TMRCA (M_pop={M_sym}, n=2)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print("Observation:")
print("- Ghost population øger TMRCA: linjer der 'parkeres' i ghost bruger ekstra tid.")
print("- Effekten minder om øget isolation — ghost fungerer som genealogisk buffer.")
print("- For høj ghost-migration dominerer ghosten genealogien.")

In [ ]:
# Kvantificer ghost-effekten: E[TMRCA] og Var som funktion af Mg
Mg_range = np.logspace(-2, 1.5, 30)
M_between = 1.0

E_ghost = []
Var_ghost = []
E_no_ghost = build_two_island(2, 0, M=M_between).expectation()

for Mg in Mg_range:
    g = build_three_pop(2, 0, M12=M_between, M21=M_between, Mg1=Mg, Mg2=Mg)
    E_ghost.append(g.expectation())
    Var_ghost.append(g.variance())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].semilogx(Mg_range, E_ghost, 'o-', label='Med ghost', lw=2)
axes[0].axhline(E_no_ghost, color='gray', linestyle='--', lw=1.5, label='Ingen ghost')
axes[0].set_xlabel('Ghost migrationsrate $M_g$')
axes[0].set_ylabel('E[TMRCA]')
axes[0].set_title('Effect af ghost population på E[TMRCA]')
axes[0].legend()

axes[1].semilogx(Mg_range, Var_ghost, 's-', color='coral', lw=2)
axes[1].set_xlabel('Ghost migrationsrate $M_g$')
axes[1].set_ylabel('Var[TMRCA]')
axes[1].set_title('Varians af TMRCA med ghost')

plt.tight_layout()
plt.show()
print("\nNøgleobservation:")
print("En ghost population med lav migration øger TMRCA markant (linjer 'fældes').")
print("Denne effekt kan FORVEKSLES med lavere M mellem de to sampledede pops!")
print("→ En ikke-samplet population bidrager med en latent genealogisk forsinkelse.")

## Tilsammen: parameterrum-eksploration

**Eksperiment:** Jeg systematisk scanner over (M, n_pop1) og
visualiserer E[TMRCA]-overfladen som et konturplot.

In [ ]:
# E[TMRCA] som funktion af M og stikprøvestørrelse n
M_scan = np.logspace(-1, 1.5, 12)
n_scan = [1, 2, 3, 4, 5]

E_surface = np.zeros((len(n_scan), len(M_scan)))

for i, n in enumerate(n_scan):
    for j, M in enumerate(M_scan):
        # Alle linjer i pop1
        g = build_two_island(n, 0, M=M)
        E_surface[i, j] = g.expectation()

fig, ax = plt.subplots(figsize=(9, 5))
for i, n in enumerate(n_scan):
    ax.semilogx(M_scan, E_surface[i, :], 'o-', label=f'n={n}', lw=1.8)

ax.set_xlabel('Migrationsrate M')
ax.set_ylabel('E[TMRCA]')
ax.set_title('E[TMRCA] som funktion af M og n (alle linjer i pop1)')
ax.legend(title='n')
plt.tight_layout()
plt.show()
print("Observation:")
print("- For høj M konvergerer alle kurver mod standardkoalescenten.")
print("- For lav M divergerer kurverne — n spiller en meget større rolle.")
print("- n=1 er speciel: kun to ekstra sekvenser samplet fra MRCA er mulig.")

## Opsummering

| Analyse | Nøgleresultat |
|---|---|
| JSFS vs. M | Lav M → private varianter dominerer; høj M → delte varianter |
| FST vs. M | FST ≈ 1/(1+2M) — verificeret mod klassisk formel |
| Asymmetrisk migration | M12 og M21 bidrager asymmetrisk til TMRCA |
| Ghost population | Øger effektivt TMRCA — kan forveksles med lav M |
| n og M interaktion | Stor n forværrer effekten af lav M |

**Næste notebook:** 05_IM_model.ipynb — Isolation-with-Migration: tidsinhomogen model.